# RhB Punctuality Analytics — Exploration
**Phase 1: Erste Analyse der Ist-Daten**

Dieses Notebook exploriert die gefilterten RhB-Daten von opentransportdata.swiss.
Ziel: Verstehen der Datenstruktur, erste Kennzahlen und Visualisierungen.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Stil
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

## 1. Daten laden

In [ ]:
# Neueste Datei im data/raw Ordner laden
raw_files = sorted(Path('../data/raw').glob('rhb_istdaten_*.csv'))
if not raw_files:
    raise FileNotFoundError('Keine Datei gefunden. Bitte zuerst fetch_istdaten.py ausführen.')

latest_file = raw_files[-1]
print(f'Lade: {latest_file}')

df = pd.read_csv(latest_file, sep=';', dtype=str, low_memory=False)

# Zeitfelder parsen
for col in ['ANKUNFTSZEIT_DT', 'AN_PROGNOSE_DT', 'ABFAHRTSZEIT_DT', 'AB_PROGNOSE_DT']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Numerische Felder
df['ANKUNFT_VERSPAETUNG_MIN'] = pd.to_numeric(df['ANKUNFT_VERSPAETUNG_MIN'], errors='coerce')
df['ABFAHRT_VERSPAETUNG_MIN'] = pd.to_numeric(df['ABFAHRT_VERSPAETUNG_MIN'], errors='coerce')
df['PUENKTLICH'] = df['PUENKTLICH'].map({'True': True, 'False': False})

print(f'Zeilen: {len(df):,} | Spalten: {len(df.columns)}')
df.head(3)

## 2. Datenstruktur verstehen

In [ ]:
# Welche Linien sind enthalten?
print('=== Linien (LINIEN_TEXT) ===')
print(df['LINIEN_TEXT'].value_counts().to_string())

In [ ]:
# Welche Verkehrsmittel?
print('=== Verkehrsmittel ===')
print(df['PRODUKT_ID'].value_counts().to_string())

In [ ]:
# Prognosestatus — wie viele echte Ist-Daten vs. Schätzungen?
print('=== Abfahrt Prognosestatus ===')
print(df['AB_PROGNOSE_STATUS'].value_counts().to_string())

In [ ]:
# Fehlende Werte prüfen
print('=== Fehlende Werte (key columns) ===')
key_cols = ['LINIEN_TEXT', 'HALTESTELLEN_NAME', 'ABFAHRTSZEIT_DT', 'AB_PROGNOSE_DT', 'ABFAHRT_VERSPAETUNG_MIN']
print(df[key_cols].isna().sum().to_string())

## 3. Kennzahlen

In [ ]:
total        = len(df)
puenktlich   = df['PUENKTLICH'].sum()
unpuenktlich = total - puenktlich
rate         = puenktlich / total * 100
median_vsp   = df['ABFAHRT_VERSPAETUNG_MIN'].median()
mean_vsp     = df['ABFAHRT_VERSPAETUNG_MIN'].mean()
p95_vsp      = df['ABFAHRT_VERSPAETUNG_MIN'].quantile(0.95)
max_vsp      = df['ABFAHRT_VERSPAETUNG_MIN'].max()
ausgefallen  = (df['FAELLT_AUS_TF'] == 'true').sum()

print(f'Betriebstag:              {df["BETRIEBSTAG"].iloc[0]}')
print(f'Haltestopps total:        {total:,}')
print(f'Pünktlich (≤3 Min):       {puenktlich:,} ({rate:.1f}%)')
print(f'Verspätet (>3 Min):       {unpuenktlich:,} ({100-rate:.1f}%)')
print(f'Median Verspätung:        {median_vsp:.1f} Min')
print(f'Durchschnitt Verspätung:  {mean_vsp:.1f} Min')
print(f'95. Perzentil:            {p95_vsp:.1f} Min')
print(f'Max. Verspätung:          {max_vsp:.1f} Min')
print(f'Ausgefallene Fahrten:     {ausgefallen}')

## 4. Visualisierungen

In [ ]:
# 4.1 Verteilung der Verspätungen
fig, ax = plt.subplots()
data = df['ABFAHRT_VERSPAETUNG_MIN'].clip(-5, 30).dropna()
ax.hist(data, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='green', linestyle='--', linewidth=1.2, label='Pünktlich')
ax.axvline(3, color='orange', linestyle='--', linewidth=1.2, label='+3 Min Grenze')
ax.set_xlabel('Verspätung bei Abfahrt (Min)')
ax.set_ylabel('Anzahl Haltestopps')
ax.set_title('Verteilung der Abfahrtsverspätungen — RhB')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Pünktlichkeit pro Linie
linie_stats = (
    df.groupby('LINIEN_TEXT')
    .agg(
        stopps=('PUENKTLICH', 'count'),
        puenktlich_rate=('PUENKTLICH', 'mean'),
        median_verspaetung=('ABFAHRT_VERSPAETUNG_MIN', 'median')
    )
    .assign(puenktlich_pct=lambda x: x['puenktlich_rate'] * 100)
    .sort_values('puenktlich_pct', ascending=True)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, max(4, len(linie_stats) * 0.4)))
colors = ['#d9534f' if v < 80 else '#f0ad4e' if v < 90 else '#5cb85c'
          for v in linie_stats['puenktlich_pct']]
bars = ax.barh(linie_stats['LINIEN_TEXT'], linie_stats['puenktlich_pct'], color=colors)
ax.axvline(84, color='gray', linestyle='--', linewidth=1, label='Tagesdurchschnitt')
ax.set_xlabel('Pünktlichkeit (%)')
ax.set_title('Pünktlichkeit nach Linie — RhB')
ax.set_xlim(0, 105)
for bar, val in zip(bars, linie_stats['puenktlich_pct']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

linie_stats

In [ ]:
# 4.3 Verspätung nach Tageszeit
df['STUNDE'] = df['ABFAHRTSZEIT_DT'].dt.hour

stunden_stats = (
    df.groupby('STUNDE')
    .agg(
        median_verspaetung=('ABFAHRT_VERSPAETUNG_MIN', 'median'),
        puenktlich_pct=('PUENKTLICH', lambda x: x.mean() * 100)
    )
    .reset_index()
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.bar(stunden_stats['STUNDE'], stunden_stats['median_verspaetung'],
        color='steelblue', width=0.7)
ax1.axhline(3, color='orange', linestyle='--', linewidth=1, label='+3 Min Grenze')
ax1.set_ylabel('Median Verspätung (Min)')
ax1.set_title('Verspätung und Pünktlichkeit nach Tageszeit — RhB')
ax1.legend()

ax2.bar(stunden_stats['STUNDE'], stunden_stats['puenktlich_pct'],
        color='#5cb85c', width=0.7)
ax2.axhline(84, color='gray', linestyle='--', linewidth=1, label='Tagesdurchschnitt')
ax2.set_ylabel('Pünktlichkeit (%)')
ax2.set_xlabel('Stunde')
ax2.set_xticks(range(0, 24))
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Top 10 unpünktlichste Haltestellen
halt_stats = (
    df.groupby('HALTESTELLEN_NAME')
    .agg(
        stopps=('PUENKTLICH', 'count'),
        puenktlich_pct=('PUENKTLICH', lambda x: x.mean() * 100),
        median_verspaetung=('ABFAHRT_VERSPAETUNG_MIN', 'median')
    )
    .query('stopps >= 5')  # Nur Haltestellen mit genug Datenpunkten
    .sort_values('puenktlich_pct', ascending=True)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(halt_stats['HALTESTELLEN_NAME'], halt_stats['puenktlich_pct'],
        color='#d9534f')
ax.set_xlabel('Pünktlichkeit (%)')
ax.set_title('Top 10 unpünktlichste Haltestellen (min. 5 Stopps) — RhB')
ax.set_xlim(0, 105)
for i, (val, med) in enumerate(zip(halt_stats['puenktlich_pct'],
                                    halt_stats['median_verspaetung'])):
    ax.text(val + 0.5, i, f'{val:.0f}% (Median: {med:.1f} Min)', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Erkenntnisse & nächste Schritte

**Erkenntnisse aus diesem Tag:**
- TODO: Ausfüllen nach Analyse

**Offene Fragen für weitere Analyse:**
- Wie entwickelt sich die Pünktlichkeit über mehrere Wochen?
- Gibt es Muster an bestimmten Wochentagen?
- Welche Linien sind chronisch unpünktlich vs. Ausreisser?
- Korreliert Verspätung mit Tageszeit / Saison?

**Nächster Schritt:** Phase 2 — Datenmodell in Supabase aufbauen und historische Daten sammeln.